# K2 · Multi-domain NER: does SciBERT work beyond AI papers?
1. **Before:** test the existing SciERC-only SciBERT (trained on AI abstracts) on SciERC **and** ScienceIE
   (Computer Science, Materials Science, Physics) → how much it drops outside AI.
2. **Train** SciBERT on SciERC + ScienceIE with shared labels **Method / Material / Task / Metric**
   (same regularisation as before, one seed 42). ~15 min on a T4.
3. **After:** test the multi-domain model the same way → the drop should shrink.

## 0. Setup (Kaggle)
Before running: right panel → **Accelerator: GPU T4 x2 (or P100)**, **Internet: On** (needs a phone-verified Kaggle account),
and **Add-ons → Secrets → add `HF_TOKEN`** (your Hugging Face write token) and tick it for this notebook.

Kaggle has no Google Drive: results go to `/kaggle/working/ResearchLens` **and** every model + its results are pushed to your
private Hugging Face account. The last cell also backs up all results to a private HF dataset.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mkavinvelavan/ResearchLens.git"
DRIVE_ROOT = "/kaggle/working/ResearchLens"          # results (Kaggle's saved output folder)
SCRATCH = "/tmp/rl"                                    # data, checkpoints, final models (large, not saved)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # one GPU, same setup as the Colab runs
for d in (DRIVE_ROOT, f"{SCRATCH}/content"):
    os.makedirs(d, exist_ok=True)
if not os.path.exists("/content"):                     # configs use /content/... paths (Colab layout)
    try:
        os.symlink(f"{SCRATCH}/content", "/content")
    except OSError as e:
        print("note: could not create /content link:", e)
os.makedirs("/content/data", exist_ok=True)

if not os.path.exists(f"{SCRATCH}/repo"):
    !git clone -q {REPO_URL} {SCRATCH}/repo
else:
    !git -C {SCRATCH}/repo pull -q
%cd {SCRATCH}/repo
!pip install -q -r requirements-colab.txt

os.environ.update({
    "RL_DRIVE_ROOT": DRIVE_ROOT, "RL_DATA_DIR": "/content/data",
    "RL_CKPT_DIR": f"{SCRATCH}/checkpoints", "RL_MODEL_DIR": f"{SCRATCH}/final_models",
    "TOKENIZERS_PARALLELISM": "false", "TQDM_DISABLE": "1",
})
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN found")
except Exception as e:
    print("WARNING: HF_TOKEN secret missing / not attached -> models will NOT be pushed to the Hub.", e)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run(module, config, run_name, seed, extra=""):
    # Train one (run, seed) unless it already finished; streams the log and stops on failure.
    import yaml
    tag = yaml.safe_load(open(config))["model_tag"]
    if os.path.exists(f"{DRIVE_ROOT}/results/{tag}/{run_name}/seed_{seed}/metrics.json"):
        print(f"skip {run_name} seed {seed} (finished)"); return
    if "--push_to_hub" in extra and not os.environ.get("HF_TOKEN"):
        extra = extra.replace("--push_to_hub", "")
    cmd = f"{sys.executable} -m {module} --config {config} --run_name {run_name} --seed {seed} {extra}"
    print(">>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if p.wait() != 0:
        raise RuntimeError(f"Training failed: {run_name} seed {seed}. Last lines:\n" + "".join(tail))

## 1. Before: the AI-only SciBERT on both domains (~2 min)

In [ ]:
!python -m researchlens.nlp.eval_ner_crossdomain --model Kavinvelavan/researchlens-scibert-ner --run_name scibert_scierc_only

## 2. Train the multi-domain SciBERT (~15 min)

In [ ]:
run("researchlens.nlp.train_ner", "configs/n1_ner_multidomain.yaml", "scibert_multidomain", 42, "--push_to_hub")

## 3. After: the multi-domain SciBERT on both domains

In [ ]:
!python -m researchlens.nlp.eval_ner_crossdomain --model /tmp/rl/final_models/n1_ner/scibert_multidomain/seed_42 --run_name scibert_multidomain

In [ ]:
import json
b = json.load(open(f"{DRIVE_ROOT}/results/n1_ner_crossdomain/scibert_scierc_only/seed_42/metrics.json"))
a = json.load(open(f"{DRIVE_ROOT}/results/n1_ner_crossdomain/scibert_multidomain/seed_42/metrics.json"))
print(f"{'test set':<28}{'AI-only':>10}{'multi-domain':>14}")
for d, name in (("scierc", "SciERC (AI)"), ("scienceie", "ScienceIE (CS/Mat/Phys)")):
    print(f"{name:<28}{b[f'test_{d}_f1_micro']*100:>9.1f}%{a[f'test_{d}_f1_micro']*100:>13.1f}%")
print(f"{'drop AI -> other fields':<28}{b['domain_drop_f1']*100:>9.1f} {a['domain_drop_f1']*100:>12.1f}  (F1 points)")

## Back up all results to Hugging Face (Kaggle has no Drive)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(); user = api.whoami()["name"]
repo = f"{user}/researchlens-results"
api.create_repo(repo, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=f"{DRIVE_ROOT}/results", path_in_repo="results", repo_id=repo, repo_type="dataset")
print(f"results backed up to https://huggingface.co/datasets/{repo} (private)")